In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))  # points to src/
from shared_modeling import load_or_create_master_split_ids, run_model_experiment

In [2]:
predictors = ['rest_dur_avg_all_Mod', 'rest_sleeptime_avg_all_Mod', 'sleep_dur_avg_all_Mod', 'sleep_sleeptime_avg_all_Mod', 'sleep_Frag_avg_all_Mod', 'sleep_WASO_avg_all_Mod', 'sleep_SE_avg_all_Mod', 'rest_sleeptime_avg_wkday_Mod']  
df = pd.read_csv('../../Data/modified/SLEEP_ACTIGRAPHY_MODIFIED.CSV', usecols=predictors + ['PublicID'])
df

,PublicID,rest_dur_avg_all_Mod,rest_sleeptime_avg_all_Mod,rest_sleeptime_avg_wkday_Mod,sleep_dur_avg_all_Mod,sleep_sleeptime_avg_all_Mod,sleep_Frag_avg_all_Mod,sleep_WASO_avg_all_Mod,sleep_SE_avg_all_Mod
0,00022M,478.916656,413.416656,399.899994,467.916656,409.666656,26.400000,58.250000,85.633331
1,00037W,577.857117,512.785706,489.299988,572.214294,510.928558,16.444286,61.285713,88.507149
2,00051F,615.900024,519.500000,537.833313,607.000000,518.500000,29.483999,88.500000,84.447998
3,00053B,460.714294,399.714294,420.000000,453.142853,396.785706,24.460001,56.357143,83.424278
4,00062A,657.857117,553.785706,578.700012,631.642883,545.142883,36.068573,86.500000,83.761429
...,...,...,...,...,...,...,...,...,...
653,17298W,484.333344,423.583344,421.000000,476.916656,422.000000,18.618334,54.916668,87.174995
654,17317V,536.642883,455.214294,434.100006,523.785706,451.714294,25.052856,72.071426,84.125717
655,17343U,535.071411,465.928558,434.000000,525.857117,465.500000,20.960001,60.357143,86.921432
656,17351V,655.000000,574.333313,536.125000,645.833313,569.583313,24.150002,76.250000,86.904999


In [3]:
df_outcomes = pd.read_csv('../../Data/Modified/Outcome.csv', usecols=['PublicID', 'MH_outcome'])

# Create the master split once and persist it for reuse in other notebooks.
split_path = 'master_split_ids.csv'
train_ids, test_ids = load_or_create_master_split_ids(df_outcomes, split_path)
df_outcomes

,PublicID,MH_outcome
0,00004O,1
1,00007I,1
2,00008G,0
3,00015J,0
4,00016H,1
...,...,...
7736,17349I,1
7737,17350A,1
7738,17351V,0
7739,17352T,1


In [4]:
df = pd.merge(df, df_outcomes, on='PublicID', how='inner')
df

,PublicID,rest_dur_avg_all_Mod,rest_sleeptime_avg_all_Mod,rest_sleeptime_avg_wkday_Mod,sleep_dur_avg_all_Mod,sleep_sleeptime_avg_all_Mod,sleep_Frag_avg_all_Mod,sleep_WASO_avg_all_Mod,sleep_SE_avg_all_Mod,MH_outcome
0,00022M,478.916656,413.416656,399.899994,467.916656,409.666656,26.400000,58.250000,85.633331,1
1,00037W,577.857117,512.785706,489.299988,572.214294,510.928558,16.444286,61.285713,88.507149,0
2,00051F,615.900024,519.500000,537.833313,607.000000,518.500000,29.483999,88.500000,84.447998,1
3,00063V,561.785706,505.214294,517.099976,551.500000,502.857147,14.527143,48.642857,89.435715,0
4,00067N,536.714294,437.571442,429.100006,510.142853,428.142853,32.689999,82.000000,79.615715,0
...,...,...,...,...,...,...,...,...,...,...
592,17298W,484.333344,423.583344,421.000000,476.916656,422.000000,18.618334,54.916668,87.174995,0
593,17317V,536.642883,455.214294,434.100006,523.785706,451.714294,25.052856,72.071426,84.125717,0
594,17343U,535.071411,465.928558,434.000000,525.857117,465.500000,20.960001,60.357143,86.921432,1
595,17351V,655.000000,574.333313,536.125000,645.833313,569.583313,24.150002,76.250000,86.904999,0


In [5]:
X = df.drop(['MH_outcome', 'PublicID'], axis=1)
y = df['MH_outcome']

train_df = df[df['PublicID'].isin(train_ids)].copy()
test_df = df[df['PublicID'].isin(test_ids)].copy()

X_train = train_df.drop(['MH_outcome', 'PublicID'], axis=1)
X_test = test_df.drop(['MH_outcome', 'PublicID'], axis=1)
y_train = train_df['MH_outcome']
y_test = test_df['MH_outcome']

y.value_counts()

MH_outcome
0    382
1    215
Name: count, dtype: int64

In [6]:
# Run the LR pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'lr',
    predictors
)

Final dataset sizes for LR (impute=False): train=487, test=110
Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters found: {'classifier__C': 0.01, 'classifier__l1_ratio': 0.25}
Best Macro F1 Score: 0.5991
Model Coefficients:
num__rest_dur_avg_all_Mod: 0.0
num__rest_sleeptime_avg_all_Mod: 0.0
num__sleep_dur_avg_all_Mod: 0.0
num__sleep_sleeptime_avg_all_Mod: 0.0
num__sleep_Frag_avg_all_Mod: 0.07635104374114655
num__sleep_WASO_avg_all_Mod: 0.05867597241140923
num__sleep_SE_avg_all_Mod: -0.10269271095257666
num__rest_sleeptime_avg_wkday_Mod: 0.0
Evaluation Metrics for LR with shared preprocessing and macro F1 grid search:
Accuracy: 0.6455
Precision: 0.6156
Recall: 0.6143
F1-score: 0.6149
ROC AUC: 0.6718


In [7]:
# Run the RF pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'rf',
    predictors
)

Final dataset sizes for RF (impute=False): train=487, test=110
Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters found: {'classifier__max_depth': 20, 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 7, 'classifier__n_estimators': 600}
Best Macro F1 Score: 0.5509
Feature Importances:
num__rest_dur_avg_all_Mod: 0.11479632831326464
num__rest_sleeptime_avg_all_Mod: 0.1182624171697689
num__sleep_dur_avg_all_Mod: 0.11415324251217787
num__sleep_sleeptime_avg_all_Mod: 0.11737080173691514
num__sleep_Frag_avg_all_Mod: 0.14102335734566113
num__sleep_WASO_avg_all_Mod: 0.13933345136536743
num__sleep_SE_avg_all_Mod: 0.14295545075986044
num__rest_sleeptime_avg_wkday_Mod: 0.11210495079698445
Evaluation Metrics for RF with shared preprocessing and macro F1 grid search:
Accuracy: 0.6091
Precision: 0.5686
Recall: 0.5643
F1-score: 0.5651
ROC AUC: 0.6229


In [8]:
# Run the XGBoost pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'xgb',
    predictors
)

Final dataset sizes for XGB (impute=False): train=487, test=110
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters found: {'classifier__colsample_bytree': 0.9, 'classifier__learning_rate': 0.05, 'classifier__max_depth': 6, 'classifier__n_estimators': 60, 'classifier__subsample': 0.5}
Best Macro F1 Score: 0.5635
Feature Importances:
num__rest_dur_avg_all_Mod: 0.1238129660487175
num__rest_sleeptime_avg_all_Mod: 0.12845925986766815
num__sleep_dur_avg_all_Mod: 0.12744201719760895
num__sleep_sleeptime_avg_all_Mod: 0.1216430738568306
num__sleep_Frag_avg_all_Mod: 0.11716464906930923
num__sleep_WASO_avg_all_Mod: 0.1252799928188324
num__sleep_SE_avg_all_Mod: 0.13113632798194885
num__rest_sleeptime_avg_wkday_Mod: 0.1250617802143097
Evaluation Metrics for XGB with shared preprocessing and macro F1 grid search:
Accuracy: 0.6091
Precision: 0.5722
Recall: 0.5696
F1-score: 0.5704
ROC AUC: 0.5979


In [9]:
# Run the SVM pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'svm',
    predictors
)

Final dataset sizes for SVM (impute=False): train=487, test=110
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters found: {'classifier__estimator__C': 0.1, 'classifier__estimator__gamma': 0.01, 'classifier__estimator__kernel': 'rbf'}
Best Macro F1 Score: 0.5990
Skipping feature-level SVM output to keep notebook output compact.
Evaluation Metrics for SVM with shared preprocessing and macro F1 grid search:
Accuracy: 0.6818
Precision: 0.6573
Recall: 0.6589
F1-score: 0.6581
ROC AUC: 0.6818
